In [6]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "Starter CSV not found."
print("Starter data ready.")

Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data ready.


# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NodeA7/FlyRank_AI/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Ranking / scoring.**

I want to rank existing pages to help a content team decide which ones to review first. Each page will have a score, reason codes, and a suggested action. The final decision remains with the editor.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

**Eventual target**: A future drop in page impressions, measured over a defined period.

**Starter-data proxy**: I use `trend_direction` to create a `decline_proxy` label: 1 for down and 0 otherwise. This label describes the current trend; it does not predict future performance.

I will not use `trend_direction` or `trend_pct` as features when predicting this proxy because that would cause data leakage.

In [8]:
import pandas as pd

csv_path = "data/raw/content_refresh_anonymized.csv"
raw = pd.read_csv(csv_path)

lane_df = raw.loc[raw["impressions_90d"].ge(100),
                   ["content_type", "impressions_90d", "days_since_last_update",
                    "avg_position", "trend_direction"]].copy()
lane_df["decline_proxy"] = lane_df["trend_direction"].astype(str).str.lower().eq("down").astype("int8")

print(f"Starter rows: {len(raw):,} | review-slice rows: {len(lane_df):,}")
print(f"Current decline proxy positives: {int(lane_df['decline_proxy'].sum()):,} ({lane_df['decline_proxy'].mean():.1%} of review slice)")
print("Proxy definition: trend_direction == 'down'")

Starter rows: 30,000 | review-slice rows: 22,006
Current decline proxy positives: 13,152 (59.8% of review slice)
Proxy definition: trend_direction == 'down'


## 3. Success metric

**Primary metric: Precision@K.**

***NOTE:K = 50 in this test***

This measures how many of the top K recommended pages meet the target. The choice of K and will depend on the team's review capacity.

I will compare the model against a fixed-rule baseline using held-out data and a future outcome. No model has been trained yet, so I am not reporting a performance score.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

Each row represents one page. For this test, I selected pages with at least 100 impressions over the last 90 days. This is an exploratory cutoff, not a final policy.

For the warehouse analysis, I may use page-by-decision-date as the unit of analysis to separate features from future outcomes.

In [10]:
# One displayed row = one content item/page in the selected review slice.
lane_df[["content_type", "impressions_90d", "days_since_last_update",
         "avg_position", "trend_direction", "decline_proxy"]].head(8)

,content_type,impressions_90d,days_since_last_update,avg_position,trend_direction,decline_proxy
0,keyword article,3803,20,10.6,down,1
1,keyword article,15320,25,20.3,down,1
2,keyword article,12581,20,36.5,down,1
3,keyword article,11751,22,6.2,stable,0
4,keyword article,19140,14,44.0,down,1
5,keyword article,3970,20,8.5,down,1
7,keyword article,1724,22,21.2,stable,0
8,keyword article,32574,20,46.0,down,1


## 5. Why ML beats a fixed rule here

ML has not yet earned its place. A fixed rule may be sufficient, but it cannot easily account for how signals such as **`impressions`, `freshness`, `average_position`**, and **`content_type`** interact.

I will use ML only if it outperforms the fixed-rule baseline on held-out data and provides useful reasons for its recommendations. Otherwise, the simpler rule may be the better option.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.